# Semana 12 — Proyecto Final
## Integrar todo lo aprendido en un proyecto completo

**Objetivo:** Demostrar dominio de los conceptos de las 11 semanas anteriores construyendo un proyecto propio que use varios elementos del ecosistema cuántico: simulación, algoritmos, circuitos reales y análisis.

**Hito verificable:** Tienes un proyecto documentado, con código funcional, análisis de resultados y reflexión sobre las limitaciones del hardware actual.

---

## Elige uno de los tres proyectos propuestos (o propón el tuyo)

### Proyecto A: Optimización cuántica con QAOA
Implementa el Quantum Approximate Optimization Algorithm para resolver MAX-CUT en un grafo pequeño.

### Proyecto B: Clasificador cuántico variacional (VQC)
Entrena un circuito cuántico parametrizado para clasificar un dataset simple.

### Proyecto C: Simulación de sistemas físicos
Simula la evolución temporal de una cadena de Ising cuántica con Trotterización.

---

Los tres proyectos están desarrollados aquí. Completa el que elijas y añade tu propio análisis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.circuit import ParameterVector
print('Librerías importadas correctamente')

---
# Proyecto A — QAOA para MAX-CUT

MAX-CUT: dado un grafo G, encontrar la partición de vértices que maximiza el número de aristas entre los dos grupos.

In [ ]:
# Grafo de ejemplo: 4 vértices, 5 aristas
# 0-1, 0-3, 1-2, 2-3, 1-3
aristas = [(0, 1), (0, 3), (1, 2), (2, 3), (1, 3)]
n_vertices = 4

def calcular_corte(bits, aristas):
    """Calcula el valor del corte para una asignación binaria."""
    corte = 0
    for u, v in aristas:
        if bits[u] != bits[v]:  # Los dos vértices en particiones diferentes
            corte += 1
    return corte

# Solución clásica bruta fuerza
mejor_corte = 0
mejor_particion = None
for i in range(2**n_vertices):
    bits = [(i >> j) & 1 for j in range(n_vertices)]
    c = calcular_corte(bits, aristas)
    if c > mejor_corte:
        mejor_corte = c
        mejor_particion = bits

print(f'Grafo con {n_vertices} vértices y {len(aristas)} aristas')
print(f'Aristas: {aristas}')
print(f'Corte máximo (fuerza bruta): {mejor_corte} aristas')
print(f'Mejor partición: {mejor_particion}  → grupo 0: {[i for i,b in enumerate(mejor_particion) if b==0]}, '
      f'grupo 1: {[i for i,b in enumerate(mejor_particion) if b==1]}')

In [ ]:
def qaoa_circuito(gamma, beta, aristas, n_vertices, p_capas=1):
    """Circuito QAOA con p capas."""
    qc = QuantumCircuit(n_vertices)
    
    # Estado inicial: superposición uniforme
    qc.h(range(n_vertices))
    
    for capa in range(p_capas):
        g = gamma[capa] if hasattr(gamma, '__len__') else gamma
        b = beta[capa]  if hasattr(beta, '__len__')  else beta
        
        # Capa de problema: e^(-i*gamma*C) para cada arista
        for u, v in aristas:
            qc.cx(u, v)
            qc.rz(2 * g, v)
            qc.cx(u, v)
        
        # Capa de mezcla: e^(-i*beta*B)
        for i in range(n_vertices):
            qc.rx(2 * b, i)
    
    return qc

def valor_esperado_corte(params, aristas, n_vertices, n_shots=4096):
    """Calcula el valor esperado del corte para parámetros dados."""
    gamma, beta = params[0], params[1]
    qc = qaoa_circuito(gamma, beta, aristas, n_vertices)
    qc.measure_all()
    
    sim = AerSimulator()
    counts = sim.run(qc, shots=n_shots).result().get_counts()
    
    # Valor esperado = suma ponderada del corte para cada resultado
    val_esperado = 0
    for bitstring, count in counts.items():
        bits = [int(b) for b in bitstring]
        corte = calcular_corte(bits, aristas)
        val_esperado += corte * count / n_shots
    
    return -val_esperado  # Negativo porque scipy minimiza

# Optimización clásica de los parámetros
x0 = [np.pi/4, np.pi/4]
print('Optimizando parámetros QAOA...')
resultado = minimize(valor_esperado_corte, x0, 
                     args=(aristas, n_vertices),
                     method='COBYLA',
                     options={'maxiter': 100, 'rhobeg': 0.5})

gamma_opt, beta_opt = resultado.x
corte_qaoa = -resultado.fun
print(f'γ óptimo: {gamma_opt:.4f}, β óptimo: {beta_opt:.4f}')
print(f'Valor esperado del corte (QAOA): {corte_qaoa:.3f}')
print(f'Corte máximo (exacto): {mejor_corte}')
print(f'Ratio de aproximación: {corte_qaoa/mejor_corte:.3f} (1.0 = óptimo)')

---
# Proyecto B — Clasificador Cuántico Variacional (VQC)

Entrenar un circuito cuántico parametrizado para clasificar puntos en 2D.

In [ ]:
# Dataset: clasificación XOR (linealmente no separable)
np.random.seed(42)
X_train = np.array([
    [0.1, 0.1], [0.2, 0.15], [0.8, 0.85], [0.9, 0.9],  # clase 0 (diagonal principal)
    [0.1, 0.9], [0.15, 0.8], [0.9, 0.1], [0.85, 0.2],  # clase 1 (diagonal secundaria)
])
y_train = np.array([0, 0, 0, 0, 1, 1, 1, 1])

def codificar_datos(x):
    """Codifica un punto 2D en rotaciones de ángulo."""
    return np.pi * x  # Mapear [0,1] → [0,π]

def vqc_circuito(x, theta):
    """Circuito VQC: capa de datos + capa variacional."""
    qc = QuantumCircuit(2)
    
    # Capa de codificación de datos
    angles = codificar_datos(x)
    qc.ry(angles[0], 0)
    qc.ry(angles[1], 1)
    qc.cx(0, 1)
    
    # Capa variacional
    qc.ry(theta[0], 0)
    qc.ry(theta[1], 1)
    qc.cx(0, 1)
    qc.ry(theta[2], 0)
    qc.ry(theta[3], 1)
    
    return qc

def predecir(x, theta):
    """Predice la clase midiendo el primer qubit."""
    qc = vqc_circuito(x, theta)
    sv = Statevector(qc)
    # P(qubit 0 = |1⟩)
    p1 = sv.probabilities([0])[1]
    return p1  # >0.5 → clase 1, <0.5 → clase 0

def coste(theta, X, y):
    """Entropía cruzada binaria."""
    total = 0
    for xi, yi in zip(X, y):
        p = predecir(xi, theta)
        p = np.clip(p, 1e-10, 1 - 1e-10)
        if yi == 1:
            total -= np.log(p)
        else:
            total -= np.log(1 - p)
    return total / len(y)

# Entrenamiento
theta0 = np.random.uniform(0, 2*np.pi, 4)
print('Entrenando VQC...')
historial_coste = []

def coste_con_historial(theta):
    c = coste(theta, X_train, y_train)
    historial_coste.append(c)
    return c

resultado_vqc = minimize(coste_con_historial, theta0, method='COBYLA',
                          options={'maxiter': 200, 'rhobeg': 0.5})
theta_opt = resultado_vqc.x

# Evaluar precisión
predicciones = [int(predecir(x, theta_opt) > 0.5) for x in X_train]
precision = np.mean(np.array(predicciones) == y_train)

print(f'Coste inicial: {historial_coste[0]:.4f}')
print(f'Coste final:   {historial_coste[-1]:.4f}')
print(f'Precisión en entrenamiento: {precision:.2%}')
print(f'Predicciones: {predicciones}')
print(f'Etiquetas:    {y_train.tolist()}')

---
# Proyecto C — Simulación de la Cadena de Ising Cuántica

Simula la evolución temporal de un sistema cuántico descrito por el Hamiltoniano de Ising.

In [ ]:
# Hamiltoniano de Ising transverso: H = -J Σ Z_i Z_{i+1} - h Σ X_i
# Para n=3 qubits en cadena abierta

def hamiltoniano_ising(n, J=1.0, h=0.5):
    """Construye el Hamiltoniano de Ising transverso matricialmente."""
    I = np.eye(2, dtype=complex)
    X = np.array([[0, 1], [1, 0]], dtype=complex)
    Z = np.array([[1, 0], [0, -1]], dtype=complex)
    
    N = 2**n
    H = np.zeros((N, N), dtype=complex)
    
    # Término de acoplamiento ZZ
    for i in range(n - 1):
        ops = [I] * n
        ops[i] = Z
        ops[i+1] = Z
        ZZ = ops[0]
        for op in ops[1:]:
            ZZ = np.kron(ZZ, op)
        H -= J * ZZ
    
    # Término de campo transverso X
    for i in range(n):
        ops = [I] * n
        ops[i] = X
        XI = ops[0]
        for op in ops[1:]:
            XI = np.kron(XI, op)
        H -= h * XI
    
    return H

def evolucionar(psi0, H, t):
    """Evolución unitaria: |ψ(t)⟩ = e^(-iHt)|ψ(0)⟩"""
    # Diagonalizar H
    eigenvalues, eigenvectors = np.linalg.eigh(H)
    # Calcular e^(-iHt)
    U_t = eigenvectors @ np.diag(np.exp(-1j * eigenvalues * t)) @ eigenvectors.conj().T
    return U_t @ psi0

# Simular para n=3 qubits
n = 3
J, h = 1.0, 0.5
H = hamiltoniano_ising(n, J, h)

# Estado inicial: todos en |0⟩
psi0 = np.zeros(2**n, dtype=complex)
psi0[0] = 1.0  # |000⟩

# Evolución temporal
tiempos = np.linspace(0, 10, 200)
Z_op = np.array([[1, 0], [0, -1]], dtype=complex)
I = np.eye(2, dtype=complex)

# Observable: magnetización del qubit 0: ⟨Z₀⟩
Z0 = np.kron(np.kron(Z_op, I), I)

magnetizacion = []
for t in tiempos:
    psi_t = evolucionar(psi0, H, t)
    m = np.real(psi_t.conj() @ Z0 @ psi_t)
    magnetizacion.append(m)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tiempos, magnetizacion, 'b-', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Tiempo t')
ax.set_ylabel('⟨Z₀⟩')
ax.set_title(f'Evolución temporal: Cadena de Ising (n={n}, J={J}, h={h})')
plt.tight_layout()
plt.savefig('ising_evolucion.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Energía del estado fundamental: {np.linalg.eigvalsh(H)[0]:.4f}')
print(f'Energía del primer estado excitado: {np.linalg.eigvalsh(H)[1]:.4f}')
print(f'Gap energético: {np.linalg.eigvalsh(H)[1] - np.linalg.eigvalsh(H)[0]:.4f}')

---
## Tu propio análisis y extensiones

Usa esta sección para añadir tu análisis propio del proyecto elegido, extensiones, y observaciones.

In [ ]:
# TU ANÁLISIS AQUÍ
# Elige uno de los tres proyectos y profundiza en él:

# Proyecto A (QAOA):
# - Aumenta a p=2 capas y compara con p=1
# - Prueba con un grafo más grande
# - Analiza el landscape del espacio de parámetros

# Proyecto B (VQC):
# - Añade más capas variacionales
# - Prueba con un dataset diferente (iris, mnist reducido)
# - Analiza el barren plateau problem

# Proyecto C (Ising):
# - Implementa la Trotterización para el operador de evolución
# - Compara la Trotterización vs. evolución exacta
# - Estudia la transición de fase cuántica variando h/J

pass

## ✅ Hito Final — Proyecto Completado

Para cerrar el curso, verifica que puedes:

### Conocimiento fundamental (semanas 1-5)
- [ ] Representar estados cuánticos como vectores y calcular probabilidades
- [ ] Aplicar puertas cuánticas y verificar unitariedad
- [ ] Entender y construir estados entrelazados
- [ ] Usar interferencia constructiva y destructiva

### Herramientas (semanas 6-7)
- [ ] Construir y simular circuitos en Qiskit
- [ ] Implementar el algoritmo de Grover

### Algoritmos avanzados (semanas 8-9)
- [ ] Implementar la QFT y entender su estructura
- [ ] Explicar el algoritmo de Shor y su impacto

### Hardware real (semanas 10-11)
- [ ] Modelar y mitigar ruido cuántico
- [ ] Ejecutar circuitos en IBM Quantum

### Proyecto final (semana 12)
- [ ] Completar un proyecto que integre varios conceptos
- [ ] Analizar los resultados críticamente
- [ ] Identificar las limitaciones del hardware actual

## Reflexión final (escribe aquí tu respuesta)

**¿Qué aplicación de la computación cuántica te parece más prometedora a corto plazo (5-10 años)?**

*(tu respuesta aquí)*

**¿Qué aspecto de la computación cuántica te gustaría explorar más profundamente?**

*(tu respuesta aquí)*

**¿Qué ha cambiado en tu comprensión desde la Semana 1?**

*(tu respuesta aquí)*

---

## Recursos para continuar

- **Qiskit Textbook**: https://learning.quantum.ibm.com/
- **Nielsen & Chuang**: *Quantum Computation and Quantum Information* (el libro de referencia)
- **arXiv quant-ph**: artículos de investigación en computación cuántica
- **IBM Quantum Network**: proyectos con hardware real
- **Quantum Computing Stack Exchange**: comunidad de preguntas y respuestas